<a href="https://colab.research.google.com/github/Misa-734/lab-ai-69/blob/main/Lab0_Colab_Gemini_TH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab วิชา Artificial Intelligence

**โดย ผศ.ดร.ณัฐโชติ พรหมฤทธิ์**

**ภาควิชาคอมพิวเตอร์ คณะวิทยาศาสตร์ มหาวิทยาลัยศิลปากร**

licensed under CC BY-NC-ND

# Lab 0 · เตรียมเครื่องมือ

**ติดตั้งของ 3 อย่าง แล้วทดสอบว่าคุยกับโมเดลรู้เรื่อง**

บน Colab เราไม่ต้องติดตั้ง Python และไม่ต้องสร้าง venv เพราะ notebook นี้คือสภาพแวดล้อมพร้อมใช้ของตัวเองอยู่แล้ว สิ่งที่ต้องทำเหลือแค่ 4 ขั้น


## ขั้นที่ 1 ขอ API key ฟรีจาก Google AI Studio

1. เปิด [aistudio.google.com](https://aistudio.google.com) แล้วล็อกอินด้วยบัญชี Google ธรรมดา
2. ไปที่ https://aistudio.google.com/apikey กดปุ่ม **Get API key** (มุมซ้ายหรือในเมนู) แล้วกด **Create API key**
3. ระบบจะโชว์ key หน้าตาประมาณ `AIza...` ให้กด copy เก็บไว้



key คือรหัสลับเทียบเท่ารหัสผ่าน ห้ามฝังในโค้ด ห้าม commit ลง git ห้ามแปะในรายงาน และห้ามแชร์ให้เพื่อน


## ขั้นที่ 2 เก็บ key ไว้ใน Colab Secrets

ที่แถบเครื่องมือด้านซ้ายของ Colab จะเห็นไอคอนรูปกุญแจ 🔑 ชื่อเมนู **Secrets**

1. กดไอคอน 🔑 แล้วกด **Add new secret**
2. ตั้งชื่อว่า `GEMINI_API_KEY` แล้ววาง key ลงช่อง Value
3. เปิดสวิตช์ **Notebook access** ให้เป็นสีฟ้า


## ขั้นที่ 3 ติดตั้งแพ็กเกจ แล้วต่อสาย key เข้าระบบ


In [1]:
# ติดตั้งแพ็กเกจ openai ซึ่งเราใช้เป็น ปลั๊กกลาง เพื่อคุยกับ Gemini เพราะ Google เปิดประตูแบบ OpenAI-compatible ไว้ให้

%pip install -q openai
print("ติดตั้งเสร็จแล้ว")

ติดตั้งเสร็จแล้ว


In [1]:
# ดึง key จาก Colab Secrets มาใส่ environment variable
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

## ขั้นที่ 4 สร้างไฟล์กลาง llm.py ที่ทุก lab จะใช้ร่วมกัน

ทุก lab ถัดจากนี้จะ `from llm import call_llm` เหมือนกันหมด เปลี่ยนผู้ให้บริการโมเดลเมื่อไหร่ก็แก้ที่ไฟล์นี้ไฟล์เดียว


In [2]:
%%writefile llm.py
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ["GEMINI_API_KEY"],
)
MODEL = "models/gemini-flash-latest"

# ---- ถ้าวันหน้าจะย้ายไป OpenAI แบบเติมเงิน แก้แค่สองบรรทัดนี้ ----
# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# MODEL = "gpt-4o-mini"

def call_llm(messages, **kwargs):
    """เรียกโมเดลหนึ่งครั้ง คืนข้อความตอบกลับ"""
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, **kwargs
    )
    return resp.choices[0].message.content


Overwriting llm.py


## ขั้นที่ 5 ทดสอบว่าทุกอย่างทำงาน


In [3]:
from llm import call_llm

answer = call_llm([{"role": "user", "content": "สวัสดี"}])
print(answer)

KeyboardInterrupt: 

### ถ้าติดปัญหา

| อาการ | สาเหตุและทางแก้ |
|---|---|
| `SecretNotFoundError` หรือขึ้นว่ายังไม่พบ secret | ยังไม่ได้ทำขั้นที่ 2 หรือสะกดชื่อ secret ไม่ตรง `GEMINI_API_KEY` หรือลืมเปิดสวิตช์ Notebook access |
| `401` หรือ `API key not valid` | key ผิดหรือ copy มาไม่ครบ กลับไปสร้าง key ใหม่ใน AI Studio แล้ววางทับใน Secrets |
| `429` หรือ `RESOURCE_EXHAUSTED` | โควตาฟรีของวันนี้หมด รอวันถัดไป หรือรันเซลล์ด้านล่างเพื่อสลับไปโมเดลที่เบากว่า |
| `404` หรือไม่รู้จักชื่อโมเดล | ชื่อรุ่นอาจเปลี่ยนตามเวลา รันเซลล์ด้านล่างเพื่อดูรายชื่อโมเดลที่ key ของคุณใช้ได้จริง |


In [5]:
# รันเมื่ออยากรู้ว่า key ของเราใช้โมเดลอะไรได้บ้าง
from llm import client

for m in client.models.list().data[:30]:
    print(m.id)

# ถ้าอยากสลับรุ่น แก้บรรทัด MODEL ในเซลล์ %%writefile llm.py
# เช่น MODEL = "gemini-2.5-flash-lite" (เบากว่า โควตาฟรีเยอะกว่า)
# แล้วรันเซลล์นั้นซ้ำ ตามด้วย restart runtime ก่อน import ใหม่

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
